# Project Completion Time Prediction


**Environment note**: This notebook is designed for Google Cloud Workbench or a similar Jupyter environment with access to BigQuery and Vertex AI.

Recommended Python version: 3.10 or later.


## Overview

This notebook is the project workflow for predicting project completion time. The current project data is stored in BigQuery, and the business target is the milestone completion date `S90`.

The workflow trains a TensorFlow/Keras DNN regression model, writes TensorBoard logs, packages the trainer for Vertex AI, then runs a Vertex AI Pipeline that registers the exported SavedModel and deploys it to a Vertex AI Endpoint.


### Objective

This project notebook implements an end-to-end MLOps flow:

- BigQuery creates reproducible train/validation/test tables.
- Cloud Storage stores exported CSV data, trainer packages, model artifacts, and TensorBoard logs.
- Local training performs a short smoke test before managed training.
- Vertex AI Pipelines runs managed custom training.
- Vertex AI Model Registry stores the exported TensorFlow SavedModel.
- Vertex AI Endpoint hosts the registered model for online prediction.

Endpoint deployment keeps one serving replica running by default. Delete the endpoint or set this section aside when you only need offline evaluation or batch prediction.


### Dataset

The project dataset is stored in BigQuery:

```text
qwilabs-asl-03-7c1aaee9a503.jindong_lin.project_data
```

Current schema used by this notebook:

- Label: `S90`
- Categorical and ID-like features: `RID`, `SID`, `PID`, `ZDP`, `GID`, `MID`
- Milestone date features: `S30`, `S44`, `S51`, `S52`, `S56`, `S68`, `S71`


## Installation

Install the packages required to execute this notebook. The bootstrap cell pins KFP-related packages to versions compatible with TensorFlow 2.18.x and Apache Beam 2.65.x, avoiding protobuf 6.x / grpcio 1.8x conflicts in managed Workbench images.


In [1]:
# VM/Workbench dependency bootstrap. Safe to rerun; repairs known protobuf/grpc conflicts.
# KFP/GCP components latest versions can pull protobuf 6.x and grpcio 1.8x, which conflict
# with TensorFlow 2.18 and Apache Beam 2.65 in the managed Workbench image used here.
import importlib.metadata as importlib_metadata
import importlib.util
import subprocess
import sys

required_modules = {
    "google.cloud.aiplatform": "google-cloud-aiplatform",
    "google.cloud.bigquery": "google-cloud-bigquery",
    "google.cloud.storage": "google-cloud-storage",
    "google_cloud_pipeline_components": "google-cloud-pipeline-components",
    "kfp": "kfp",
    "db_dtypes": "db-dtypes",
    "tensorflow": "tensorflow",
    "tensorboard": "tensorboard",
}

# Keep this set compatible with TensorFlow 2.18.x and Apache Beam 2.65.x:
# - tensorflow requires protobuf >=3.20.3,<6
# - apache-beam requires protobuf >=3.20.3,<6 and grpcio <1.66
# - newer google-cloud-pipeline-components/kfp releases may install protobuf 6.x generated code
# - datasets 2.14.x requires fsspec[http] <2023.9.0
pinned_packages = [
    "protobuf==4.25.9",
    "grpcio==1.65.5",
    "grpcio-status==1.62.3",
    "kfp==2.7.0",
    "kfp-pipeline-spec==0.3.0",
    "kfp-server-api==2.0.5",
    "google-cloud-pipeline-components==2.19.0",
    "fsspec[http]==2023.6.0",
]

missing_packages = [
    package for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]

install_packages = pinned_packages + [
    package for package in missing_packages if package not in pinned_packages
]

print("Installing/repairing compatible packages:", install_packages)
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--upgrade",
    *install_packages,
])

versions_to_show = [
    "protobuf",
    "grpcio",
    "grpcio-status",
    "kfp",
    "kfp-pipeline-spec",
    "google-cloud-pipeline-components",
    "tensorflow",
    "apache-beam",
    "datasets",
    "fsspec",
]
print("Installed versions:")
for package in versions_to_show:
    try:
        print(f"  {package}: {importlib_metadata.version(package)}")
    except importlib_metadata.PackageNotFoundError:
        print(f"  {package}: not installed")

print("If protobuf/grpc were changed in this run, restart the notebook kernel before continuing.")


Installing/repairing compatible packages: ['protobuf==4.25.9', 'grpcio==1.65.5', 'grpcio-status==1.62.3', 'kfp==2.7.0', 'kfp-pipeline-spec==0.3.0', 'kfp-server-api==2.0.5', 'google-cloud-pipeline-components==2.19.0', 'fsspec[http]==2023.6.0']
Installed versions:
  protobuf: 4.25.9
  grpcio: 1.65.5
  grpcio-status: 1.62.3
  kfp: 2.7.0
  kfp-pipeline-spec: 0.3.0
  google-cloud-pipeline-components: 2.19.0
  tensorflow: 2.19.1
  apache-beam: not installed
  datasets: not installed
  fsspec: 2023.6.0
If protobuf/grpc were changed in this run, restart the notebook kernel before continuing.


In [2]:
import math
import os
import warnings
from datetime import datetime

# Keep local smoke tests on CPU in Workbench environments where CUDA libraries
# may be partially present but not usable.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorboard
from google import api_core
from google.cloud import aiplatform, bigquery


%load_ext tensorboard
%load_ext google.cloud.bigquery
warnings.filterwarnings("ignore")


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


The google.cloud.bigquery extension is already loaded. To reload it, use:
  %reload_ext google.cloud.bigquery


## Set Up

### Set up your Google Cloud project

1. Select or create a Google Cloud project.
2. Make sure billing is enabled for the project.
3. Enable the required APIs:
   - BigQuery API
   - Vertex AI API
   - Cloud Storage API
4. If you are running locally, install and initialize the Google Cloud SDK.


#### Set your project ID


In [3]:
# set up project ID
PROJECT = !gcloud config get-value project  # noqa: E999
PROJECT_ID = PROJECT[0]
BUCKET = PROJECT_ID
REGION = "us-central1"

OUTDIR = f"gs://{BUCKET}/jindong_lin/data"

%env PROJECT_ID=$PROJECT_ID
%env BUCKET=$BUCKET
%env REGION=$REGION
%env OUTDIR=$OUTDIR

env: PROJECT_ID=qwiklabs-asl-03-7c1aaee9a503
env: BUCKET=qwiklabs-asl-03-7c1aaee9a503
env: REGION=us-central1
env: OUTDIR=gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data


#### Region


In [4]:
%%bash
set -e

# Enable APIs needed by this notebook. If they are already enabled, this is a no-op.
gcloud services enable   aiplatform.googleapis.com   bigquery.googleapis.com   storage.googleapis.com

# Create the project bucket if it does not exist yet.
if ! gsutil ls -b gs://$BUCKET >/dev/null 2>&1; then
  gsutil mb -l $REGION gs://$BUCKET
fi

gsutil ls -Lb gs://$BUCKET | grep "gs://\|Location"
echo $REGION


Operation "operations/acat.p2-192763454488-6989588a-0946-452f-ae7e-c70e2daf8b3a" finished successfully.


gs://qwiklabs-asl-03-7c1aaee9a503/ :
	Location type:			region
	Location constraint:		US-CENTRAL1
us-central1


In [5]:
%%bash
gcloud config set project $PROJECT_ID
gcloud config set ai/region $REGION

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Updated property [ai/region].


## Initialize Google Cloud clients


In [6]:
bq_client = bigquery.Client(project=PROJECT_ID)

## Load data from BigQuery


In [7]:
DATASET_ID = "jindong_lin"
TABLE_ID = "project_data"
BQ_TABLE = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

query = f"""
SELECT
  RID,
  SID,
  PID,
  ZDP,
  GID,
  MID,
  S30,
  S44,
  S51,
  S52,
  S56,
  S68,
  S71,
  S90
FROM `{BQ_TABLE}`
WHERE S30 IS NOT NULL
  AND S90 IS NOT NULL
"""

df = bq_client.query(query).to_dataframe()
print(df.shape)
df.head()


(120023, 14)


,RID,SID,PID,ZDP,GID,MID,S30,S44,S51,S52,S56,S68,S71,S90
0,W,134992294,701173562,Z145,G1,M8,2000-12-05,2022-08-02,2022-07-15,2000-09-13,2022-07-22,NaT,2001-04-06,2022-09-26
1,N,210990414,700069790,Z148,G3,M5,2006-04-05,NaT,NaT,NaT,NaT,NaT,2006-04-27,2023-01-09
2,S,588990170,700130200,Z144,G12,M7,2008-01-30,2020-03-04,2022-05-11,2021-10-01,2022-05-10,2022-08-16,2022-08-16,2022-08-18
3,W,464990117,700102074,Z3,G6,M12,2008-09-09,NaT,NaT,2008-12-09,2009-02-18,2009-10-26,NaT,2021-10-07
4,W,346990318,700146135,Z3,G1,M4,2009-03-16,NaT,NaT,2008-11-25,NaT,2009-04-27,NaT,2021-10-07


## Inspect schema and missing values


In [8]:
df.info()
missing_summary = df.isna().mean().sort_values(ascending=False).to_frame("missing_rate")
missing_summary


<class 'pandas.DataFrame'>
RangeIndex: 120023 entries, 0 to 120022
Data columns (total 14 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   RID     120023 non-null  str   
 1   SID     120023 non-null  Int64 
 2   PID     120023 non-null  Int64 
 3   ZDP     85585 non-null   str   
 4   GID     120023 non-null  str   
 5   MID     119912 non-null  str   
 6   S30     120023 non-null  dbdate
 7   S44     48897 non-null   dbdate
 8   S51     119718 non-null  dbdate
 9   S52     19105 non-null   dbdate
 10  S56     35539 non-null   dbdate
 11  S68     21077 non-null   dbdate
 12  S71     76678 non-null   dbdate
 13  S90     120023 non-null  dbdate
dtypes: Int64(2), dbdate(8), str(4)
memory usage: 13.9 MB


,missing_rate
S52,0.840822
S68,0.824392
S56,0.703898
S44,0.592603
S71,0.361139
ZDP,0.286928
S51,0.002541
MID,0.000925
RID,0.000000
SID,0.000000


## Managed training with Vertex AI Training Service

This section:

1. Creates reproducible BigQuery train, validation, and test tables.
2. Exports the tables as CSV files to Cloud Storage.
3. Moves the training code into a normal Python package.
4. Trains a Keras DNN model locally as a smoke test.
5. Submits the same package to Vertex AI Custom Training.
6. Opens TensorBoard logs for local and Vertex training runs.


In [9]:
# Use the same project environment values configured above.

dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
BQ_LOCATION = bq_client.get_dataset(dataset_ref).location
print("BigQuery dataset location:", BQ_LOCATION)

# Reuse the project bucket and OUTDIR configured near the beginning of the notebook.
BUCKET_URI = f"gs://{BUCKET}"
FEATURE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.project_completion_features"
TRAIN_TABLE = f"{PROJECT_ID}.{DATASET_ID}.project_completion_train"
VALID_TABLE = f"{PROJECT_ID}.{DATASET_ID}.project_completion_valid"
TEST_TABLE = f"{PROJECT_ID}.{DATASET_ID}.project_completion_test"
DATA_GCS_PREFIX = f"{OUTDIR}/project_completion"

print("Training data export path:", DATA_GCS_PREFIX)

BigQuery dataset location: us-central1
Training data export path: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion


In [10]:
BUCKET_URI

'gs://qwiklabs-asl-03-7c1aaee9a503'

### BigQuery location note

The notebook detects the dataset location automatically instead of assuming a fixed multi-region. This avoids BigQuery extract failures when the dataset and Cloud Storage bucket are not in compatible locations.

```python
BQ_LOCATION = bq_client.get_dataset(dataset_ref).location
```

Use this detected value for both BigQuery query jobs and `bq extract`.


### Create BigQuery train, validation, and test tables

The split uses `FARM_FINGERPRINT`, following the training notebook pattern. This makes the split deterministic and avoids a different random split every time the notebook runs.


In [11]:
feature_sql = f"""
CREATE OR REPLACE TABLE `{FEATURE_TABLE}` AS
SELECT
  DATE_DIFF(DATE(S90), DATE(S30), DAY) AS days_to_S90,
  COALESCE(CAST(RID AS STRING), 'UNKNOWN') AS RID,
  COALESCE(CAST(ZDP AS STRING), 'UNKNOWN') AS ZDP,
  COALESCE(CAST(GID AS STRING), 'UNKNOWN') AS GID,
  COALESCE(CAST(MID AS STRING), 'UNKNOWN') AS MID,
  DATE_DIFF(DATE(S44), DATE(S30), DAY) AS days_S44_from_S30,
  DATE_DIFF(DATE(S51), DATE(S30), DAY) AS days_S51_from_S30,
  DATE_DIFF(DATE(S52), DATE(S30), DAY) AS days_S52_from_S30,
  DATE_DIFF(DATE(S56), DATE(S30), DAY) AS days_S56_from_S30,
  DATE_DIFF(DATE(S68), DATE(S30), DAY) AS days_S68_from_S30,
  DATE_DIFF(DATE(S71), DATE(S30), DAY) AS days_S71_from_S30,
  COALESCE(CAST(SID AS STRING), '') AS SID,
  COALESCE(CAST(PID AS STRING), '') AS PID,
  'unused' AS key
FROM `{BQ_TABLE}`
WHERE S30 IS NOT NULL
  AND S90 IS NOT NULL
  AND DATE_DIFF(DATE(S90), DATE(S30), DAY) >= 0
"""

split_sql = f"""
CREATE OR REPLACE TABLE `{TRAIN_TABLE}` AS
SELECT * FROM `{FEATURE_TABLE}`
WHERE ABS(MOD(FARM_FINGERPRINT(CONCAT(SID, '-', PID)), 100)) < 80;

CREATE OR REPLACE TABLE `{VALID_TABLE}` AS
SELECT * FROM `{FEATURE_TABLE}`
WHERE ABS(MOD(FARM_FINGERPRINT(CONCAT(SID, '-', PID)), 100)) BETWEEN 80 AND 89;

CREATE OR REPLACE TABLE `{TEST_TABLE}` AS
SELECT * FROM `{FEATURE_TABLE}`
WHERE ABS(MOD(FARM_FINGERPRINT(CONCAT(SID, '-', PID)), 100)) >= 90;
"""

bq_client.query(feature_sql, location=BQ_LOCATION).result()
bq_client.query(split_sql, location=BQ_LOCATION).result()

ROW_COUNTS = {}
for table in [FEATURE_TABLE, TRAIN_TABLE, VALID_TABLE, TEST_TABLE]:
    ROW_COUNTS[table] = bq_client.get_table(table).num_rows
    print(table, ROW_COUNTS[table])

TRAIN_ROW_COUNT = ROW_COUNTS[TRAIN_TABLE]
VALID_ROW_COUNT = ROW_COUNTS[VALID_TABLE]
TEST_ROW_COUNT = ROW_COUNTS[TEST_TABLE]

if TRAIN_ROW_COUNT == 0:
    raise ValueError(f"Training table is empty: {TRAIN_TABLE}")
if VALID_ROW_COUNT == 0:
    raise ValueError(f"Validation table is empty: {VALID_TABLE}")
if TEST_ROW_COUNT == 0:
    print(f"Warning: test table is empty: {TEST_TABLE}. Consider using a less sparse split key.")


qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_features 118721
qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_train 94913
qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_valid 11850
qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_test 11958


### Export the tables as CSV files

The exported CSV files follow the same order as the training table. The first column is the label `days_to_S90`.


In [12]:
from google.api_core.exceptions import NotFound
from google.cloud import storage

print("BQ_LOCATION:", BQ_LOCATION)
print("DATA_GCS_PREFIX:", DATA_GCS_PREFIX)

storage_client = storage.Client(project=PROJECT_ID)
bucket_name = DATA_GCS_PREFIX.replace("gs://", "").split("/", 1)[0]
print("Export bucket:", bucket_name)

# BigQuery extract jobs require the destination bucket to be compatible with the
# BigQuery dataset location. Create the project bucket if it does not exist.
try:
    bucket = storage_client.get_bucket(bucket_name)
    print("Bucket location:", bucket.location)
except NotFound:
    bucket = storage_client.bucket(bucket_name)
    bucket.location = BQ_LOCATION
    bucket = storage_client.create_bucket(bucket)
    print("Created bucket:", bucket.name, "location:", bucket.location)

# Remove old exported CSV files with Python client instead of gsutil/gcloud shell.
prefix = DATA_GCS_PREFIX.replace(f"gs://{bucket_name}/", "").rstrip("/") + "/"
old_blobs = list(storage_client.list_blobs(bucket_name, prefix=prefix))
for blob in old_blobs:
    blob.delete()
print(f"Deleted {len(old_blobs)} old objects under gs://{bucket_name}/{prefix}")

extract_jobs = [
    (TRAIN_TABLE, f"{DATA_GCS_PREFIX}/train/project-train-*.csv"),
    (VALID_TABLE, f"{DATA_GCS_PREFIX}/valid/project-valid-*.csv"),
    (TEST_TABLE, f"{DATA_GCS_PREFIX}/test/project-test-*.csv"),
]

job_config = bigquery.job.ExtractJobConfig(
    destination_format=bigquery.DestinationFormat.CSV,
    field_delimiter=",",
    print_header=False,
)

for table, destination_uri in extract_jobs:
    print("Exporting", table, "to", destination_uri)
    job = bq_client.extract_table(
        table,
        destination_uri,
        job_config=job_config,
        location=BQ_LOCATION,
    )
    job.result()
    print("Done:", job.job_id)

# List exported files without wildcard expansion.
for split in ["train", "valid", "test"]:
    split_prefix = f"{prefix}{split}/"
    blobs = list(storage_client.list_blobs(bucket_name, prefix=split_prefix))
    print(f"{split}: {len(blobs)} files")
    for blob in blobs[:5]:
        print(" ", f"gs://{bucket_name}/{blob.name}", blob.size)
    if not blobs:
        raise RuntimeError(f"No files exported for {split}. Check the extract job errors above.")



BQ_LOCATION: us-central1
DATA_GCS_PREFIX: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion
Export bucket: qwiklabs-asl-03-7c1aaee9a503
Bucket location: US-CENTRAL1
Deleted 3 old objects under gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/
Exporting qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_train to gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/train/project-train-*.csv
Done: 70b663cf-f045-4e41-a30e-36945bb2ada9
Exporting qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_valid to gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/valid/project-valid-*.csv
Done: 6338bf43-ad5b-4581-84a4-d9198b998aaa
Exporting qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_test to gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/test/project-test-*.csv
Done: 5ce7ab86-17e6-48bb-b45c-012729d98ead
train: 1 files
  gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/da

## Make code compatible with Vertex AI Training Service

The notebook code now becomes a regular Python package that can run the same way locally and on Vertex AI Custom Training.


In [13]:
#! rm -rf project_completion
#! mkdir -p project_completion/trainer
#! touch project_completion/trainer/__init__.py

### Create `model.py`

This module reads the exported CSV files with `tf.data`, builds a Keras embedding DNN regression model, evaluates it in original day units, writes TensorBoard logs, and exports both a `.keras` model file and a serving SavedModel.

This version improves the earlier smoke-test DNN in two ways:

- Categorical features (`RID`, `ZDP`, `GID`, `MID`) use learned embeddings instead of sparse one-hot vectors.
- The regression label is scaled during training with `label_scale`, then metrics are converted back to days for comparison with the tree model.

`SID` and `PID` are exported for deterministic splitting and traceability, but are intentionally not used as direct model features.


In [14]:
# make it in projects seperate

### Create `task.py`

This is the command-line entry point used by local testing and Vertex AI Custom Training.


In [15]:
# make it in projects seperate

### Run the trainer locally from the notebook

This local run is a short smoke test that trains for 3 epochs and writes TensorBoard scalar logs to Cloud Storage. The trainer uses local checkpoint/backup directories during `model.fit()` to avoid slow per-epoch model writes to GCS; final model artifacts and metrics are written to GCS only after training completes.


In [16]:
LOCAL_OUTPUT_DIR = f"{BUCKET_URI}/project_completion/local_embedding_dnn_test"

LOCAL_BATCH_SIZE = 64
LOCAL_EPOCHS = 3

# If this cell is run out of order, recover row counts from the BigQuery tables
# created above instead of failing with NameError.
if "TRAIN_ROW_COUNT" not in globals() or "VALID_ROW_COUNT" not in globals():
    TRAIN_ROW_COUNT = bq_client.get_table(TRAIN_TABLE).num_rows
    VALID_ROW_COUNT = bq_client.get_table(VALID_TABLE).num_rows

if TRAIN_ROW_COUNT == 0 or VALID_ROW_COUNT == 0:
    raise ValueError(
        f"Train/valid tables must be non-empty before local training. "
        f"Got train={TRAIN_ROW_COUNT}, valid={VALID_ROW_COUNT}."
    )

# Keep the notebook smoke test short. Vertex training below still uses the full
# computed step counts.
LOCAL_TRAIN_STEPS_PER_EPOCH = min(100, max(1, math.ceil(TRAIN_ROW_COUNT / LOCAL_BATCH_SIZE)))
LOCAL_VALIDATION_STEPS = min(50, max(1, math.ceil(VALID_ROW_COUNT / LOCAL_BATCH_SIZE)))

print("Local train rows:", TRAIN_ROW_COUNT)
print("Local valid rows:", VALID_ROW_COUNT)
print("Local smoke train_steps_per_epoch:", LOCAL_TRAIN_STEPS_PER_EPOCH)
print("Local smoke validation_steps:", LOCAL_VALIDATION_STEPS)

! cd project_completion && CUDA_VISIBLE_DEVICES=-1 TF_CPP_MIN_LOG_LEVEL=2 python -m trainer.task   --train_data_path "{DATA_GCS_PREFIX}/train/project-train-*"   --eval_data_path "{DATA_GCS_PREFIX}/valid/project-valid-*"   --output_dir "{LOCAL_OUTPUT_DIR}"   --batch_size {LOCAL_BATCH_SIZE}   --epochs {LOCAL_EPOCHS}   --train_steps_per_epoch {LOCAL_TRAIN_STEPS_PER_EPOCH}   --validation_steps {LOCAL_VALIDATION_STEPS}   --hidden_units "128 64 32"   --learning_rate 0.001   --dropout_rate 0.2   --label_scale 365.0


Local train rows: 94913
Local valid rows: 11850
Local smoke train_steps_per_epoch: 100
Local smoke validation_steps: 50
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/jupyter/project_completion/trainer/task.py", line 5, in <module>
    from trainer import model
  File "/home/jupyter/project_completion/trainer/model.py", line 12, in <module>
    import keras
ModuleNotFoundError: No module named 'keras'


#### Open TensorBoard for the local smoke test

The local smoke test writes event files under `LOCAL_OUTPUT_DIR/tensorboard`. If the training cell has finished, open TensorBoard from the notebook with the next cell.


In [17]:
%tensorboard --logdir {LOCAL_OUTPUT_DIR}/tensorboard --port 8000

ERROR: Could not find `tensorboard`. Please ensure that your PATH
contains an executable `tensorboard` program, or explicitly specify
the path to a TensorBoard binary by setting the `TENSORBOARD_BINARY`
environment variable.

## Run your training package on Vertex AI


In [18]:
"""Build a source distribution for Vertex AI custom training."""

# project_completion/setup.py

'Build a source distribution for Vertex AI custom training.'

In [19]:
! cd project_completion && python setup.py sdist --formats=gztar

PACKAGE_URI = f"{BUCKET_URI}/project_completion/packages/project_completion_trainer-0.1.tar.gz"
! gsutil cp project_completion/dist/project_completion_trainer-0.1.tar.gz {PACKAGE_URI}
print(PACKAGE_URI)


running sdist
running egg_info
writing project_completion_trainer.egg-info/PKG-INFO
writing dependency_links to project_completion_trainer.egg-info/dependency_links.txt
writing requirements to project_completion_trainer.egg-info/requires.txt
writing top-level names to project_completion_trainer.egg-info/top_level.txt
reading manifest file 'project_completion_trainer.egg-info/SOURCES.txt'
writing manifest file 'project_completion_trainer.egg-info/SOURCES.txt'

running check
creating project_completion_trainer-0.1
creating project_completion_trainer-0.1/project_completion_trainer.egg-info
creating project_completion_trainer-0.1/trainer
copying files to project_completion_trainer-0.1...
copying setup.py -> project_completion_trainer-0.1
copying project_completion_trainer.egg-info/PKG-INFO -> project_completion_trainer-0.1/project_completion_trainer.egg-info
copying project_completion_trainer.egg-info/SOURCES.txt -> project_completion_trainer-0.1/project_completion_trainer.egg-info
copying

### Vertex AI training image note

Use a TensorFlow Vertex AI prebuilt training image for the Keras DNN trainer. The image version should match the TensorFlow version used by the packaged trainer.


### Submit Custom Job using the Vertex AI SDK

This mirrors the Python SDK submission style from the training notebook. The job always writes TensorBoard event files. To also sync those logs into **Vertex AI TensorBoard**, set `USE_VERTEX_TENSORBOARD = True` and provide a `SERVICE_ACCOUNT` that the submitter is allowed to act as.


In [20]:
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=f"{BUCKET_URI}/project_completion/staging")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
MODEL_OUTPUT_DIR = f"{BUCKET_URI}/project_completion/trained_dnn_model_{timestamp}"

BATCH_SIZE = 64
EPOCHS = 10
if "TRAIN_ROW_COUNT" not in globals() or "VALID_ROW_COUNT" not in globals():
    TRAIN_ROW_COUNT = bq_client.get_table(TRAIN_TABLE).num_rows
    VALID_ROW_COUNT = bq_client.get_table(VALID_TABLE).num_rows

TRAIN_STEPS_PER_EPOCH = max(1, math.ceil(TRAIN_ROW_COUNT / BATCH_SIZE))
VALIDATION_STEPS = max(1, math.ceil(VALID_ROW_COUNT / BATCH_SIZE))
HIDDEN_UNITS = "128 64 32"
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.2
LABEL_SCALE = 365.0

# The end-to-end MLOps path below uses Vertex AI Pipelines.
# Keep this direct CustomJob path optional so Run all does not train twice.
RUN_DIRECT_VERTEX_CUSTOM_JOB = True
USE_VERTEX_TENSORBOARD = False
SERVICE_ACCOUNT = None  # Example: "vertex-training-sa@PROJECT_ID.iam.gserviceaccount.com"
TENSORBOARD_RESOURCE_NAME = None

print("Vertex train rows:", TRAIN_ROW_COUNT)
print("Vertex valid rows:", VALID_ROW_COUNT)
print("Vertex train_steps_per_epoch:", TRAIN_STEPS_PER_EPOCH)
print("Vertex validation_steps:", VALIDATION_STEPS)

if USE_VERTEX_TENSORBOARD:
    tensorboard_display_name = "project-completion-tensorboard"
    existing_tensorboards = aiplatform.Tensorboard.list(
        filter=f'display_name="{tensorboard_display_name}"',
        project=PROJECT_ID,
        location=REGION,
    )
    if existing_tensorboards:
        vertex_tensorboard = existing_tensorboards[0]
    else:
        vertex_tensorboard = aiplatform.Tensorboard.create(
            display_name=tensorboard_display_name,
            project=PROJECT_ID,
            location=REGION,
        )
    TENSORBOARD_RESOURCE_NAME = vertex_tensorboard.resource_name
    print("Vertex AI TensorBoard:", TENSORBOARD_RESOURCE_NAME)

args = [
    "--train_data_path", f"{DATA_GCS_PREFIX}/train/project-train-*",
    "--eval_data_path", f"{DATA_GCS_PREFIX}/valid/project-valid-*",
    "--output_dir", MODEL_OUTPUT_DIR,
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--train_steps_per_epoch", str(TRAIN_STEPS_PER_EPOCH),
    "--validation_steps", str(VALIDATION_STEPS),
    "--hidden_units", HIDDEN_UNITS,
    "--learning_rate", str(LEARNING_RATE),
    "--dropout_rate", str(DROPOUT_RATE),
    "--label_scale", str(LABEL_SCALE),
]

worker_pool_specs = [
    {
        "machine_spec": {"machine_type": "n1-standard-4"},
        "replica_count": 1,
        "python_package_spec": {
            "executor_image_uri": "us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest",
            "package_uris": [PACKAGE_URI],
            "python_module": "trainer.task",
            "args": args,
        },
    }
]

if RUN_DIRECT_VERTEX_CUSTOM_JOB:
    training_job = aiplatform.CustomJob(
        display_name=f"project_completion_dnn_training_{timestamp}",
        worker_pool_specs=worker_pool_specs,
        staging_bucket=f"{BUCKET_URI}/project_completion/staging",
    )

    submit_kwargs = {}
    if TENSORBOARD_RESOURCE_NAME:
        if not SERVICE_ACCOUNT:
            raise ValueError("Set SERVICE_ACCOUNT before enabling Vertex AI TensorBoard sync.")
        submit_kwargs["tensorboard"] = TENSORBOARD_RESOURCE_NAME
        submit_kwargs["service_account"] = SERVICE_ACCOUNT

    training_job.submit(**submit_kwargs)
    print("Submitted direct Vertex AI custom training job")
    print("Model output directory:", MODEL_OUTPUT_DIR)
else:
    print("Skipping direct CustomJob. The next section runs the managed Vertex AI Pipeline instead.")

Vertex train rows: 94913
Vertex valid rows: 11850
Vertex train_steps_per_epoch: 1484
Vertex validation_steps: 186
Creating CustomJob
CustomJob created. Resource name: projects/192763454488/locations/us-central1/customJobs/941198978626617344
To use this CustomJob in another session:
custom_job = aiplatform.CustomJob.get('projects/192763454488/locations/us-central1/customJobs/941198978626617344')
View Custom Job:
https://console.cloud.google.com/agent-platform/locations/us-central1/training/941198978626617344?project=192763454488
Submitted direct Vertex AI custom training job
Model output directory: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/trained_dnn_model_20260609_100255


#### Open TensorBoard for the Vertex AI job

If `USE_VERTEX_TENSORBOARD` is `False`, the trainer writes event files to `MODEL_OUTPUT_DIR/tensorboard`, and you can open them directly from the notebook.

If `USE_VERTEX_TENSORBOARD` is `True`, Vertex AI sets `AIP_TENSORBOARD_LOG_DIR` for the training job and automatically uploads logs to the Vertex AI TensorBoard resource passed to `training_job.submit(tensorboard=...)`. In that case, open the TensorBoard from the Vertex AI console.


In [21]:
%tensorboard --logdir {MODEL_OUTPUT_DIR}/tensorboard --port 8000


ERROR: Could not find `tensorboard`. Please ensure that your PATH
contains an executable `tensorboard` program, or explicitly specify
the path to a TensorBoard binary by setting the `TENSORBOARD_BINARY`
environment variable.

## End-to-end MLOps with Vertex AI Pipelines

This section turns the existing project training package into a Kubeflow/Vertex AI Pipeline. The pipeline runs the same `trainer.task` package on Vertex AI Custom Training, imports the exported TensorFlow SavedModel, registers it in Vertex AI Model Registry, creates an Endpoint, and deploys the model for online serving.

The pipeline is the default managed path for this notebook. The direct CustomJob section above is optional and skipped by default to avoid duplicate training jobs when you run the notebook from top to bottom.


In [22]:
PIPELINE_ROOT = f"{BUCKET_URI}/project_completion/kfp_pipeline"
PIPELINE_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
PIPELINE_MODEL_OUTPUT_DIR = f"{BUCKET_URI}/project_completion/kfp_trained_dnn_model_{PIPELINE_TIMESTAMP}"
PIPELINE_YAML = "project_completion_kfp_pipeline.yaml"
PIPELINE_PY = "project_completion/pipeline_vertex/project_completion_kfp_pipeline.py"

TRAIN_DATA_PATH = f"{DATA_GCS_PREFIX}/train/project-train-*"
EVAL_DATA_PATH = f"{DATA_GCS_PREFIX}/valid/project-valid-*"

BATCH_SIZE = globals().get("BATCH_SIZE", 64)
EPOCHS = globals().get("EPOCHS", 10)
HIDDEN_UNITS = globals().get("HIDDEN_UNITS", "128 64 32")
LEARNING_RATE = globals().get("LEARNING_RATE", 0.001)
DROPOUT_RATE = globals().get("DROPOUT_RATE", 0.2)
LABEL_SCALE = globals().get("LABEL_SCALE", 365.0)
SERVING_MACHINE_TYPE = globals().get("SERVING_MACHINE_TYPE", "n1-standard-2")
SERVING_MIN_REPLICA_COUNT = globals().get("SERVING_MIN_REPLICA_COUNT", 1)
SERVING_MAX_REPLICA_COUNT = globals().get("SERVING_MAX_REPLICA_COUNT", 1)
if "TRAIN_ROW_COUNT" not in globals() or "VALID_ROW_COUNT" not in globals():
    TRAIN_ROW_COUNT = bq_client.get_table(TRAIN_TABLE).num_rows
    VALID_ROW_COUNT = bq_client.get_table(VALID_TABLE).num_rows

TRAIN_STEPS_PER_EPOCH = max(1, math.ceil(TRAIN_ROW_COUNT / BATCH_SIZE))
VALIDATION_STEPS = max(1, math.ceil(VALID_ROW_COUNT / BATCH_SIZE))
PIPELINE_NAME = "project-completion-dnn"
MODEL_DISPLAY_NAME = f"{PIPELINE_NAME}-{PIPELINE_TIMESTAMP}"
ENDPOINT_DISPLAY_NAME = f"{PIPELINE_NAME}-endpoint-{PIPELINE_TIMESTAMP}"

os.environ.update({
    "PROJECT_ID": PROJECT_ID,
    "REGION": REGION,
    "PIPELINE_ROOT": PIPELINE_ROOT,
    "PACKAGE_URI": PACKAGE_URI,
    "TRAIN_DATA_PATH": TRAIN_DATA_PATH,
    "EVAL_DATA_PATH": EVAL_DATA_PATH,
    "MODEL_OUTPUT_DIR": PIPELINE_MODEL_OUTPUT_DIR,
    "TRAIN_STEPS_PER_EPOCH": str(TRAIN_STEPS_PER_EPOCH),
    "VALIDATION_STEPS": str(VALIDATION_STEPS),
    "BATCH_SIZE": str(BATCH_SIZE),
    "EPOCHS": str(EPOCHS),
    "HIDDEN_UNITS": HIDDEN_UNITS,
    "LEARNING_RATE": str(LEARNING_RATE),
    "DROPOUT_RATE": str(DROPOUT_RATE),
    "LABEL_SCALE": str(LABEL_SCALE),
    "PIPELINE_NAME": PIPELINE_NAME,
    "MODEL_DISPLAY_NAME": MODEL_DISPLAY_NAME,
    "ENDPOINT_DISPLAY_NAME": ENDPOINT_DISPLAY_NAME,
    "SERVING_MACHINE_TYPE": SERVING_MACHINE_TYPE,
    "SERVING_MIN_REPLICA_COUNT": str(SERVING_MIN_REPLICA_COUNT),
    "SERVING_MAX_REPLICA_COUNT": str(SERVING_MAX_REPLICA_COUNT),
})

print("Pipeline root:", PIPELINE_ROOT)
print("Pipeline model output dir:", PIPELINE_MODEL_OUTPUT_DIR)
print("Training package:", PACKAGE_URI)
print("Train data:", TRAIN_DATA_PATH)
print("Eval data:", EVAL_DATA_PATH)
print("Endpoint display name:", ENDPOINT_DISPLAY_NAME)
print("Serving machine type:", SERVING_MACHINE_TYPE)


Pipeline root: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/kfp_pipeline
Pipeline model output dir: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/kfp_trained_dnn_model_20260609_100256
Training package: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/packages/project_completion_trainer-0.1.tar.gz
Train data: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/train/project-train-*
Eval data: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/valid/project-valid-*
Endpoint display name: project-completion-dnn-endpoint-20260609_100256
Serving machine type: n1-standard-2


In [23]:
#!mkdir -p project_completion/pipeline_vertex


In [24]:
"""Vertex AI Pipeline for the project completion DNN trainer."""

#project_completion/pipeline_vertex/project_completion_kfp_pipeline.py

'Vertex AI Pipeline for the project completion DNN trainer.'

In [25]:
from kfp import compiler
import importlib.util

spec = importlib.util.spec_from_file_location("project_completion_kfp_pipeline", PIPELINE_PY)
pipeline_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline_module)

compiler.Compiler().compile(
    pipeline_func=pipeline_module.create_pipeline,
    package_path=PIPELINE_YAML,
)

print("Compiled pipeline YAML:", PIPELINE_YAML)


Compiled pipeline YAML: project_completion_kfp_pipeline.yaml


In [26]:
RUN_VERTEX_PIPELINE = True

if RUN_VERTEX_PIPELINE:
    aiplatform.init(
        project=PROJECT_ID,
        location=REGION,
        staging_bucket=f"{BUCKET_URI}/project_completion/staging",
    )

    pipeline_job = aiplatform.PipelineJob(
        display_name=f"project_completion_dnn_pipeline_{PIPELINE_TIMESTAMP}",
        template_path=PIPELINE_YAML,
        pipeline_root=PIPELINE_ROOT,
        enable_caching=True,
    )

    # sync=True makes Run all wait until training, model upload, endpoint creation, and deployment finish.
    pipeline_job.run(sync=True)
    print("Pipeline resource name:", pipeline_job.resource_name)
    print("Model output directory:", PIPELINE_MODEL_OUTPUT_DIR)
    print("TensorBoard logs:", f"{PIPELINE_MODEL_OUTPUT_DIR}/tensorboard")
    print("Endpoint display name:", ENDPOINT_DISPLAY_NAME)
else:
    print("RUN_VERTEX_PIPELINE is False; compiled the pipeline but did not submit it.")


Creating PipelineJob
PipelineJob created. Resource name: projects/192763454488/locations/us-central1/pipelineJobs/project-completion-dnn-pipeline-20260609100256
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/192763454488/locations/us-central1/pipelineJobs/project-completion-dnn-pipeline-20260609100256')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/project-completion-dnn-pipeline-20260609100256?project=192763454488
PipelineJob projects/192763454488/locations/us-central1/pipelineJobs/project-completion-dnn-pipeline-20260609100256 current state:
PIPELINE_STATE_RUNNING
PipelineJob projects/192763454488/locations/us-central1/pipelineJobs/project-completion-dnn-pipeline-20260609100256 current state:
PIPELINE_STATE_RUNNING
PipelineJob projects/192763454488/locations/us-central1/pipelineJobs/project-completion-dnn-pipeline-20260609100256 current state:
PIPELINE_STATE_RUNNING
PipelineJob proje

### Open TensorBoard for the pipeline run

After the pipeline finishes, TensorBoard event files are under `PIPELINE_MODEL_OUTPUT_DIR/tensorboard`.


In [27]:
%tensorboard --logdir {PIPELINE_MODEL_OUTPUT_DIR}/tensorboard --port 8005


ERROR: Could not find `tensorboard`. Please ensure that your PATH
contains an executable `tensorboard` program, or explicitly specify
the path to a TensorBoard binary by setting the `TENSORBOARD_BINARY`
environment variable.

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
